Information Extraction

In [ ]:
import os
import time
from glob import glob
import pandas as pd
from tqdm import tqdm
import tiktoken
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain, SequentialChain
from langchain.chat_models import ChatOpenAI

# === Configuration ===
TRANSCRIPTS_DIR = "Transcripts_Deepgram_nova-2_order by patient"
MODEL_NAME = "gpt-4o"

# Cost per 1K tokens for GPT-4o
INPUT_COST_PER_1K  = 0.005   # USD
OUTPUT_COST_PER_1K = 0.015   # USD

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"] 

# === LLM Setup ===
chat_model = ChatOpenAI(
    model=MODEL_NAME,
    temperature=0.0,
    openai_api_key=OPENAI_API_KEY
)

# Step 1: Generate Multiple Types of Knowledge
generate_knowledge_prompt = PromptTemplate(
    input_variables=["conversation"],
    template="""
    あなたは、会話の中から健康関連データを抽出するために必要な多様な知識を生成するAIアシスタントです。以下のカテゴリについて、各種知識を生成してください：

    **1. 文脈知識 (Contextual Knowledge)**:
    - 体温（摂氏°C）: 通常36.1〜37.2°C。
    - 血圧（mmHg、収縮期/拡張期）: 通常90/60 mmHg〜120/80 mmHg。
    - 脈拍（1分あたりの回数）: 60〜100回。
    - 呼吸（1分あたりの回数）: 12〜20回。
    - SPO2（%）: 通常95%〜100%。
    - 排便（有/無）:
      言及があれば「有」または「無」と出力してください。言及がない場合、問題がなさそうなら「有」、便秘や下痢の示唆があれば「無」や具体的な状態を示してください。
    - 食欲（有/無）:
      言及があれば「有」または「無」と出力してください。言及がない場合、問題がなさそうなら「有」、食欲低下の示唆があれば「無」や具体的な状態を示してください。

    **2. ヒューリスティック知識 (Heuristic Knowledge)**:
    - 最近の身体活動がある場合、脈拍や呼吸が一時的に上昇する可能性。
    - SPO2が低い場合、呼吸やその他の問題を疑うべき。

    **3. 言語バリエーション (Linguistic Variations)**:
    - 「体温」は「熱」「お熱」などの表現でも言及される可能性。
    - 日本語の体温表現には「六度キューブ」「七度二分」などが含まれ、以下のルールに従って数値に変換してください:「六度キューブ」 → 36.9°C
    - 血圧の数値は「上が120、下が80」など異なるフォーマットで表現されることがある。

    **4. 推論ガイドライン (Inference Guidelines)**:
    - 明確な数値がない場合、典型的な範囲から合理的に値を推測してください。
    - 文脈に基づいて特定の健康傾向を予測してください（例: 食欲がない場合、栄養不足の可能性）。

    会話:
    {conversation}
    """
)

# Step 2: Extract Key Information
extract_key_info_prompt = PromptTemplate(
    input_variables=["conversation", "generated_knowledge"],
    template="""
    あなたは、看護師と高齢患者の会話から健康関連データを抽出するAIアシスタントです。以下のカテゴリについて、会話と生成された知識を基に具体的な値を提供してください。明確に言及されていない場合でも、文脈や生成された知識を活用して値を推測してください。「不明」や「なし」という回答は避け、必ず具体的な値を提供してください。また、出力は日本語と英語で行ってください。

    **カテゴリ**:
    - 体温（摂氏°C）
    - 血圧（mmHg、収縮期/拡張期）
    - 脈拍（1分あたりの回数）
    - 呼吸（1分あたりの回数）
    - SPO2（%）
    - 排便（有/無）
    - 食欲（有/無）

    **推論ポリシー**:
    1. 明確な数値が会話に含まれている場合、それを使用してください。
    2. 明確な言及がない場合は、生成された知識や会話の文脈を基に合理的に値を推測してください。
    3. 必ず具体的な値を提供し、「不明」や「言及なし」などの回答は避けてください。
    4. バイナリ値（排便や食欲など）は、文脈や生成された知識を基に「有」または「無」を選んでください。
    5. 排便については、患者が排便や便通に明確に言及している場合、「有」と答えてください。言及がない場合は、会話の文脈や健康状態に基づき、排便が問題ないと推測できる場合は「有」、問題があると推測される場合は「無」と答えてください。
    6. 食欲についても、患者が食欲に明確に言及している場合、「有」と答えてください。言及がない場合は、会話の文脈に基づいて推論し、問題がないと推測される場合は「有」、問題があると推測される場合は「無」と答えてください。
    7. 出力は必ず日本語と英語の両方で行ってください。

    **出力例**:
    - 体温 / Body Temperature: 36.4°C
      明確に言及 / Mentioned explicitly: "お熱が六度四分ですね" / (Your temperature is 36.4 degrees)
    - 血圧 / Blood Pressure: 124/60 mmHg
      明確に言及 / Mentioned explicitly: "血圧が124の60" / (Blood pressure is 124/60)
    - 脈拍 / Pulse: 66回/分
      明確に言及 / Mentioned explicitly: "脈が66です" / (Pulse is 66)
    - 呼吸 / Breath: 18回/分
      明確に言及 / Mentioned explicitly: "きゅう18%の脈が66です" / (Breath is 18 and pulse is 66)
    - SPO2: 98%
      推測 / Inferred: Based on typical elderly SPO2 values as no explicit mention in the conversation.
    - 排便 / Defecation: 有 / Having
      明確に言及 / Mentioned explicitly: "便がありました" / (There was defecation)
    - 食欲 / Appetite: 有 / Having
      推測 / Inferred: Based on the absence of any complaint regarding appetite.

    **生成された知識**:
    {generated_knowledge}

    **会話**:
    {conversation}
    """
)

# === Chain Setup ===
generate_knowledge_chain = LLMChain(llm=chat_model, prompt=generate_knowledge_prompt, output_key="generated_knowledge")
extract_key_info_chain = LLMChain(llm=chat_model,prompt=extract_key_info_prompt,output_key="key_info")

# Combine Chains
chain = SequentialChain(
    chains=[generate_knowledge_chain, extract_key_info_chain],
    input_variables=["conversation"],
    output_variables=["generated_knowledge", "key_info"], 
    verbose=True 
)

# === Token Cost Functions ===
def count_tokens(text, model=MODEL_NAME):
    encoding = tiktoken.encoding_for_model(model)
    return len(encoding.encode(text))

def calculate_chain_cost(input_text, output_text):
    input_tokens = count_tokens(input_text)
    output_tokens = count_tokens(output_text)
    input_cost = (input_tokens / 1000) * INPUT_COST_PER_1K
    output_cost = (output_tokens / 1000) * OUTPUT_COST_PER_1K
    return input_tokens, output_tokens, input_cost + output_cost

# === Process All Transcript Files ===
txt_files = sorted(glob(os.path.join(TRANSCRIPTS_DIR, "P*", "*.txt")))
print(f"Found {len(txt_files)} files: {txt_files}")
results = []
total_input_tokens = total_output_tokens = 0
total_cost = 0.0

start_time = time.time()

for file_path in tqdm(txt_files, desc="Processing transcripts"):
    file_start_time = time.time()  # Per-file timing
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            conversation_text = f.read().strip()

        result = chain({"conversation": conversation_text})
        knowledge = result["generated_knowledge"]
        key_info = result["key_info"]

        print("processing: ",file_path)
        print("result:", key_info)

        in1, out1, cost1 = calculate_chain_cost(conversation_text, knowledge)
        in2, out2, cost2 = calculate_chain_cost(knowledge, key_info)

        results.append({
            "file": file_path,
            "input_tokens": in1 + in2,
            "output_tokens": out1 + out2,
            "cost_usd": round(cost1 + cost2, 4),
            "key_info": key_info
        })

        total_input_tokens += in1 + in2
        total_output_tokens += out1 + out2
        total_cost += cost1 + cost2

    except Exception as e:
        results.append({
            "file": file_path,
            "error": str(e)
        })

end_time = time.time()
processing_time = round(end_time - start_time, 2)

# === Save & Show Results ===
df = pd.DataFrame(results)
df.to_csv("extracted_health_info_results_fs-gkp_deepgram.csv", index=False)

print("\n✅ Finished processing all files.")
print(f"Total Files: {len(txt_files)}")
print(f"Total Input Tokens: {total_input_tokens}")
print(f"Total Output Tokens: {total_output_tokens}")
print(f"Total Cost (USD): ${round(total_cost, 4)}")
print(f"Total Processing Time: {processing_time} seconds")

Summarize Second Prompt -- USE THIS

In [ ]:
import os
import re
import pandas as pd
from pathlib import Path
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate, HumanMessagePromptTemplate

from langchain.chains import LLMChain

import time
import tiktoken

# Totals
total_input_tokens = 0
total_output_tokens = 0
total_cost = 0.0
total_time = 0.0

# Cost per 1K tokens for GPT-4o
INPUT_COST_PER_1K  = 0.005   # USD
OUTPUT_COST_PER_1K = 0.015   # USD

llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0.3,
    openai_api_key=os.environ["OPENAI_API_KEY"]
)

FORM_ROOT = Path("data/Visiting Form/Visiting Form - Order by Patient")
TRANSCRIPT_ROOT = Path("Transcripts_Whisper_order by patient")

def find_visiting_form_excels(root: Path) -> list[str]:
    exts = {".xls", ".xlsx"}
    files = [str(p) for p in root.rglob("*") if p.is_file() and p.suffix.lower() in exts and not p.name.startswith("~$")]
    files.sort()
    return files

form_files = find_visiting_form_excels(FORM_ROOT)
print(f"Discovered {len(form_files)} form files.")

def safe_stem(path_str: str) -> str:
    stem = Path(path_str).stem
    stem = re.sub(r'[\\/:*?"<>|]+', '_', stem)
    stem = re.sub(r'\s+', ' ', stem).strip()
    return stem

def load_form_sheets(file_path):
    excel = pd.ExcelFile(file_path)
    return {sheet: pd.read_excel(file_path, sheet_name=sheet) for sheet in excel.sheet_names}

def extract_category_names(sheet_df, section_header):
    section_row = sheet_df[sheet_df.apply(lambda row: row.astype(str).str.contains(section_header).any(), axis=1)].index
    if section_row.empty:
        return []
    section_row = section_row[0]
    category_pattern = r'【(.*?)】'
    categories = []
    for idx in range(section_row + 1, sheet_df.shape[0]):
        row = sheet_df.iloc[idx]
        matches = re.findall(category_pattern, ' '.join(row.dropna().astype(str)))
        if matches:
            categories.extend(matches)
        if row.isna().all():
            break
    return categories

# === Map a form path to the matching transcript .txt path ===
def map_form_to_transcript_path(form_path: str) -> Path:
    """
    Example:
      form:       Visiting Form/Visiting Form - Order by Patient/P1/abc.xlsx
      transcript: Experiment/Transcripts_Deepgram_nova-2_order by patient/P1/abc.txt
    """
    form_p = Path(form_path)
    rel = form_p.relative_to(FORM_ROOT)            # e.g., P1/abc.xlsx
    rel_txt = rel.with_suffix(".txt")              #       P1/abc.txt
    return (TRANSCRIPT_ROOT / rel_txt)

def run_chain(chain, variables, label=""):
    # Build chat messages from the chain's ChatPromptTemplate
    messages = chain.prompt.format_messages(**variables)
    response = chain.llm.invoke(messages)

    # Text content
    text = response.content

    # Usage (OpenAI chat models via LangChain expose this)
    usage = response.response_metadata.get("token_usage", {}) or {}
    in_tok = usage.get("prompt_tokens", 0)
    out_tok = usage.get("completion_tokens", 0)

    # Cost
    cost = (in_tok / 1000.0) * INPUT_COST_PER_1K + (out_tok / 1000.0) * OUTPUT_COST_PER_1K

    return text, in_tok, out_tok, cost

def read_transcript_text(txt_path: Path) -> str:
    # Try UTF-8 first, then CP932/Shift-JIS fallback (common for Japanese Windows excel/notes)
    for enc in ("utf-8", "cp932", "shift_jis", "utf-8-sig"):
        try:
            with open(txt_path, "r", encoding=enc, errors="strict") as f:
                text = f.read()
            # Normalize BOM and whitespace lightly
            return text.replace("\ufeff", "").strip()
        except Exception:
            continue
    # Last resort with replacement (won't crash the pipeline)
    try:
        with open(txt_path, "r", encoding="utf-8", errors="replace") as f:
            return f.read().replace("\ufeff", "").strip()
    except Exception as e:
        print(f"[WARN] Failed to read transcript: {txt_path} ({e})")
        return ""
    
# Prompt 1: Extract Key Information
extract_prompt = ChatPromptTemplate.from_messages([
HumanMessagePromptTemplate.from_template("""
次の会話記録から、患者の状態に関するすべての関連する健康情報を詳細に抽出します。次の詳細を含めます。

1. 利用者の状態（状態・トピックス）
症状、行動、気分の変化、観察された身体的または精神的状態
会話中に明示的に言及された、または文脈から推測された状態

2. 実施した看護・リハビリテーションの内容
訪問中に看護師によって提供されたケアの名前をすべてリストします（例：バイタルサインの確認、状態の観察、部分的な洗浄、薬の指示、水分摂取の指示）。
会話の中で明示的に言及されていない場合は、可能な限り文脈から推測し、同じまたは類似した名前を使用します。

3. コメント (次回の予定またはテストの予定):
「コメント」欄には、【受診予定】【日付】【医療機関または場所】の形式で次回の受診または予約を記入してください。例：【来院予定】1月中旬～下旬、大橋郷脳神経外科。

トランスクリプト全体からすべての関連情報を詳細に抽出します。
会話の記録:
{transcript}

抽出された情報:
""")
])

# Prompt 2: Categorize Information

# Prompt 3: Generate Contextual Knowledge
contextual_knowledge_prompt = ChatPromptTemplate.from_messages([
    HumanMessagePromptTemplate.from_template("""
   分類された情報を基に、必要な構造に沿って精査し、専門的な洞察を加え、文脈的な関連性を確保してください。

    ### 構造に関する知識:
    - 各カテゴリの情報が、例に示された形式に準拠するようにしてください。
    - 全体的に専門的で一貫したトーンを維持してください。

    ### 専門知識:
    - 看護や医療用語（例: バイタルチェック、服薬指導、ストレッチ、社会資源活用指導、介護指導、ＡＤＬ指導）を適切に使用し、記載されたすべての行動が看護実践に沿っていることを確認してください。
    - 服薬遵守、バイタルサイン、特定の看護介入などの重要なポイントを強調してください。

    ### 文脈に関する知識:
    - 記録から暗示される詳細（例: 症状の原因や行動パターン）を推測してください。
    - 文脈的に重要な詳細を強調してください（例: 薬の飲み忘れや体重測定の拒否の理由）。
    - 患者の行動と潜在的な看護アクションまたは推奨事項を結び付けてください。

    分類された情報:
    {categorized_data}

    精査された情報（構造、専門、文脈）:
    """)
])

# Prompt 4: Summarize by Categories
summarize_prompt = ChatPromptTemplate.from_messages([
    HumanMessagePromptTemplate.from_template("""
確認した情報を使用して、次の形式で最終要約を作成してください。専門的で簡潔な口調を保ち、提供されている例は出力の構造とスタイルを示すための参考であり、入力情報に基づいて要約を適切に生成してください。

注意点:
1. 動的カテゴリは、以下の {categories} に基づいています。これらのカテゴリに基づいて情報を分類し、それぞれのカテゴリについて詳細な説明を提供してください。もし情報が明記されていない場合でも、文脈から推測できる内容を詳細に記載してください。
2. 静的カテゴリ (<処置内容・訓練内容>、<備考>、<外部連絡・伝達事項>、<内部連絡・伝達事項>) は常に出力に含め、関連情報を豊富に書いてください。

例文は参考のためのみ記載されており、その内容をそのまま使用しないこと。出力は入力情報に基づいて完全にカスタマイズされている必要があります。
各カテゴリで、入力情報の詳細を反映した正確で具体的な記述を作成してください。
形式やスタイルは例に従いますが、入力データを反映しない情報を含めないでください。
「実施した看護・リハビリテーションの内容」セクションには、次の例に基づき、同じまたは類似の名前を用いてリスト形式で記載してください。例: バイタルチェック、状態観察、全身清拭、オムツ交換、パット交換、創処置、褥瘡処置、キャビロン塗布、呼吸訓練、介護指導、ＡＤＬ指導、療養指導、ストマ管理、薬物管理、在宅酸素管理。
                                             
最終要約形式:
以下の形式とスタイルを参考に、入力情報に基づいて独自の要約を作成してください。
                                             
1. 利用者の状態（状態・トピックス）
【体調】風邪症状なし。右大腿痛増強あるが、カロナール内服は無し。死腔の増大の影響もあるかもしれないと説明し、洗浄後自己にてドレナージ促した。昼食は餅を摂取し、胃部不快はなし。
【褥瘡他】（右大転子部）コアグラ極少量。クリーム色アイテル中等量。（臀部）クリーム色アイテル中等量。左側の一部で処置中も含め出血が見られたが疼痛なし。他部位は悪化なし。セルフケアでの包交は就寝前の1回のみとのこと。２か所小さな糜爛あり、出血少量あり。
【ストマ】便貯留（性3-4、量2－3)あり。ストマ３時方向に軽度糜爛あり。周囲発赤あるも自覚症状なし。
<処置内容・訓練内容>
【褥瘡処置】（右大転子部）イソジンゲル＋不織布＋パット。周囲皮膚はワセリン+アズノール塗布。（臀部）イソジンゲル＋不織布＋穴あきビニール貼布。 糜爛部にイソジン+不織布+穴あきビニール貼付（肛門部）イソジンゲル＋ガーゼ貼布。  (右大腿部裏面）発赤部にアズノール塗布＋ガーゼ貼付。 （両鼠経部）イソジンゲル+アズノール塗布。
【ストマ管理】洗浄後にパウチ交換。３時方向にアクトシン塗布、近接部にパウダー塗布。
【保湿剤塗布】全身にヘパリンフォーム、乾燥部（両腰部）にワセリン塗布追加。両足背〜足趾にかけてワセリン塗布。 左足趾間はガーゼ貼用。
<備考>
綿棒お渡し済み、代金預かりあり。
<外部連絡・伝達事項>
MCSで報告予定。
<内部連絡・伝達事項>

2.実施した看護・リハビリテーションの内容
生活援助:バイタルチェック、状態観察、全身清拭、ひげそり、爪切り、水分摂取介助、精神的援助、介護者支援
排泄援助:パウチ交換、オムツ交換、パット交換	
処置:外用薬塗布、創処置、褥瘡処置、キャビロン塗布
リハビリ:呼吸訓練
指導:観察方法指導、合併症予防指導、排便調整指導、食事指導、服薬指導、社会資源活用指導、緊急時対応指導、在宅酸素療法指導、介護指導、ＡＤＬ指導、療養指導、水分摂取指導
管理:ストマ管理、薬物管理、在宅酸素管理

3. コメント:
【北九州総合病院受診】2/13(火)
※（予定）薬効や日内変動を把握するため。1/27より、次回受診日まで病状日誌を開始。

---
確認した情報:
{generated_knowledge}
---
                                             
### 最終要約形式:
以下の形式を参考に、入力情報に基づいて独自の要約を作成してください。

1. 利用者の状態（状態・トピックス）
各サブカテゴリ (例: [身体状態]、[褥瘡]、[ストーマ] など) について、以下のように説明してください:
1. 利用者の状態（状態・トピックス）
- {categories}: 各カテゴリーに関連するすべての詳細情報を長文で記述してください。症状、行動、身体的・精神的状態をより深く掘り下げて、観察された変化や看護師の対応、推測される次のステップなども含めてください。症状の進行や、文脈から推測される問題点に関しても推論を行ってください。
- <処置内容・訓練内容>: 関連する治療の詳細やトレーニングの詳細を書く。
- <備考>: 特別または重要な備考を書く。
- <外部連絡・伝達事項>: 外部機関や利害関係者の連絡先を書く。
- <内部連絡・伝達事項>: 内部コミュニケーションのための連絡先を記入する。

2. 実施した看護/リハビリテーションの内容
※以下のサブカテゴリに分類してください。ケアの名前のみ記載。
    - 生活援助:
    - 排泄援助:
    - 処置:
    - リハビリ:
    - 指導:
    - 管理:

3. コメント
- 次回の予定や重要な情報を入力します:
    形式: [予定検査] [日付]、[医療機関または場所]
    例: [北九州総合診療科] 2/13 (火) * (予定) 薬の効能と日々の変動を把握するため

                                             
作業内容:
上記の形式に従い、確認した情報に基づいて最終サマリーを生成してください。
""")
])


# Initialize Prompt Chains
extract_chain = LLMChain(llm=llm, prompt=extract_prompt)
contextual_knowledge_chain = LLMChain(llm=llm, prompt=contextual_knowledge_prompt)
summarize_chain = LLMChain(llm=llm, prompt=summarize_prompt)

grand_input_tokens = 0
grand_output_tokens = 0
grand_cost = 0.0
grand_time = 0.0


# Main Loop
for file_path in form_files:
    start_time = time.time()
    try:
        form = load_form_sheets(file_path)
        sheet1 = form.get('Sheet1')
        if sheet1 is None or sheet1.empty:
            print(f"Sheet1 not found or is empty in {file_path}")
            continue

        categories = extract_category_names(sheet1, '利用者の状態（病状・トピックス）')
        print(f"[{file_path}] Extracted Categories: {categories}")

        # Dynamically generate the category-specific part of the prompt
        category_details = "\n".join([f"- {category}: 記載内容" for category in categories])
        print("category_details",category_details)

        # Load Transcript
        transcript_path = map_form_to_transcript_path(file_path)
        if not transcript_path.exists():
            print(f"[WARN] Transcript not found for form → {transcript_path}")
            full_transcript = ""
        else:
            full_transcript = read_transcript_text(transcript_path)

        
        extracted_data, in1, out1, c1 = run_chain(extract_chain, {"transcript": full_transcript}, "extract")
        print("Extracted Data:", extracted_data) 
        
        dynamic_categorize_prompt = f"""
抽出された情報を次の動的カテゴリと静的カテゴリに分類してください。提供されたカテゴリは動的であり、すべてのカテゴリを反映する必要があります。

### 注意点:
### 注:
1. 動的カテゴリは、以下の {categories} に基づいています。これらのカテゴリに基づいて情報を分類し、それぞれのカテゴリについて詳細な説明を提供してください。もし情報が明記されていない場合でも、文脈から推測できる内容を詳細に記載してください。
2. 静的カテゴリ (<処置内容・訓練内容>、<備考>、<外部連絡・伝達事項>、<内部連絡・伝達事項>) は常に出力に含め、関連情報を豊富に書いてください。
3. 「実施した看護・リハビリテーションの内容」セクションでは、会話から得られた情報をもとに、以下のケア項目をすべて網羅するようにしてください。もし情報が不足している場合は、推測に基づいて類似したケア項目を補完してください。以下のケア項目を参考に、すべての内容を詳細に記載してください。  
- 生活援助、排泄援助、処置、リハビリ、指導、管理に関するすべてのケア内容をリスト形式で提供してください。
---

### カテゴリ:
#### 1. 利用者の状態（状態・トピックス）
以下のカテゴリについて、それぞれの詳細情報を記載してください。以下のすべてのカテゴリを含めてください:
- {category_details}:各カテゴリに関連する具体的な情報を提供するか、情報が明示的に記載されていない場合は推測してください。
- <処置内容・訓練内容>:該当する処置内容と訓練内容を記載してください。
- <備考>:特記事項や重要なメモを記載してください。
- <外部連絡・伝達事項>:外部機関や関係者への連絡内容を記載してください。
- <内部連絡・伝達事項>:内部で共有するべき事項を記載してください。

#### 2. 実施した看護・リハビリテーションの内容
※以下のサブカテゴリに分類してください。ケアの名前のみ記載。
    - 生活援助: バイタルチェック、状態観察、全身清拭、爪切りなど
    - 排泄援助: パウチ交換、オムツ交換、パット交換など
    - 処置: 外用薬塗布、創処置、褥瘡処置など
    - リハビリ: 呼吸訓練など
    - 指導: 観察方法指導、排便調整指導、食事指導など
    - 管理: ストマ管理、薬物管理、在宅酸素管理など

#### 3. コメント:
次回の来院または検査の予定を推測どおりに記載してください。

---

### 抽出された情報:
{extracted_data}

---

### 分類された情報:
1. **利用者の状態（状態・トピックス）**
以下のカテゴリについて、それぞれの詳細情報を記載してください。以下のすべてのカテゴリを含めてください:
- {category_details}: 該当する情報:カテゴリごとに内容を記載
- <処置内容・訓練内容>: 
- <備考>: 
- <外部連絡・伝達事項>: 
- <内部連絡・伝達事項>: 

2. **実施した看護・リハビリテーションの内容**
以下のカテゴリごとに、実施されたすべての看護内容を名前のみリスト形式で記載してください。
※ケアの名前のみ記載
- 生活援助: 
- 排泄援助: 
- 処置: 
- リハビリ: 
- 指導: 
- 管理: 

3. コメント: 
"""

        dynamic_categorize_chain = LLMChain(llm=llm, prompt=ChatPromptTemplate.from_template(dynamic_categorize_prompt))        
        
        categorized_data, in2, out2, c2 = run_chain(dynamic_categorize_chain, {"extracted_data": extracted_data, "categories": categories}, "categorize")
        print("Categorized Data:", categorized_data)

        generated_knowledge, in3, out3, c3 = run_chain(contextual_knowledge_chain, {"categorized_data": categorized_data}, "contextual")
        print("Generated Knowledge:", generated_knowledge)

        summary, in4, out4, c4 = run_chain(summarize_chain, {"generated_knowledge": generated_knowledge, "categories": categories}, "summarize")
        print("Summary:", summary)

        file_input_tokens = in1 + in2 + in3 + in4
        file_output_tokens = out1 + out2 + out3 + out4
        file_cost = c1 + c2 + c3 + c4

        # Add to grand totals
        grand_input_tokens += file_input_tokens
        grand_output_tokens += file_output_tokens
        grand_cost += file_cost
        
        # Save Results
        out_name = f"summarized_transcript_fs-gkp_{safe_stem(file_path)}.xlsx"
        results = {
            "Transcript": full_transcript,
            "Extracted Data": extracted_data,
            "Categorized Data": categorized_data,
            "Generated Knowledge": generated_knowledge,
            "Generated Summary": summary
        }
        pd.DataFrame([results]).to_excel(out_name, index=False)

    except Exception as e:
        print(f"Error processing {file_path}: {e}")
    
    finally:
        elapsed = time.time() - start_time
        grand_time += elapsed
        print(f"[{file_path}] Processing time: {elapsed:.2f} sec")


print("\n=== Summary ===")
print(f"Total input tokens: {grand_input_tokens}")
print(f"Total output tokens: {grand_output_tokens}")
print(f"Total estimated cost: ${grand_cost:.4f}")
print(f"Total processing time: {grand_time:.2f} sec")


